<a href="https://colab.research.google.com/github/rah-ds/Cloud-Autoscaling-using-RL/blob/bmcgregor%2Fsimulator-refactor/notebooks/Experiment_DQN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Experiment 1 - DQN

In [10]:
raw_url_utilities = 'https://raw.githubusercontent.com/rah-ds/Cloud-Autoscaling-using-RL/refs/heads/bmcgregor/simulator-refactor/scripts/DQN_Utils.py'
local_filename_utilities = 'DQN_Utils.py'
!wget -O {local_filename_utilities} {raw_url_utilities}

--2025-11-26 00:14:35--  https://raw.githubusercontent.com/rah-ds/Cloud-Autoscaling-using-RL/refs/heads/bmcgregor/simulator-refactor/scripts/DQN_Utils.py
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.110.133, 185.199.109.133, 185.199.111.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.110.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 4687 (4.6K) [text/plain]
Saving to: ‘DQN_Utils.py’

DQN_Utils.py        100%[===================>]   4.58K  --.-KB/s    in 0s      

2025-11-26 00:14:36 (53.9 MB/s) - ‘DQN_Utils.py’ saved [4687/4687]



In [11]:
raw_url_dqn_agent = 'https://raw.githubusercontent.com/rah-ds/Cloud-Autoscaling-using-RL/refs/heads/bmcgregor/simulator-refactor/scripts/DQN_Agent.py'
local_filename_dqn_agent = 'DQN_Agent.py'
!wget -O {local_filename_dqn_agent} {raw_url_dqn_agent}

--2025-11-26 00:14:37--  https://raw.githubusercontent.com/rah-ds/Cloud-Autoscaling-using-RL/refs/heads/bmcgregor/simulator-refactor/scripts/DQN_Agent.py
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 2609 (2.5K) [text/plain]
Saving to: ‘DQN_Agent.py’

DQN_Agent.py        100%[===================>]   2.55K  --.-KB/s    in 0s      

2025-11-26 00:14:37 (25.8 MB/s) - ‘DQN_Agent.py’ saved [2609/2609]



In [12]:

raw_url = 'https://raw.githubusercontent.com/rah-ds/Cloud-Autoscaling-using-RL/refs/heads/bmcgregor/simulator-refactor/scripts/autoscaling_env.py'



# Replace with the desired local filename for your script
local_filename = 'autoscaling_env.py'

!wget -O {local_filename} {raw_url}

--2025-11-26 00:14:43--  https://raw.githubusercontent.com/rah-ds/Cloud-Autoscaling-using-RL/refs/heads/bmcgregor/simulator-refactor/scripts/autoscaling_env.py
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.111.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 7669 (7.5K) [text/plain]
Saving to: ‘autoscaling_env.py’

autoscaling_env.py  100%[===================>]   7.49K  --.-KB/s    in 0.001s  

2025-11-26 00:14:43 (11.4 MB/s) - ‘autoscaling_env.py’ saved [7669/7669]



In [13]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [14]:
import pandas as pd

df_usage = pd.read_csv('/content/drive/MyDrive/df_usage.csv')
display(df_usage.head())

,time_window,avg_cpu,avg_mem,active_machines
0,1970-01-01 00:05:00+00:00,0.006623,0.004912,9525
1,1970-01-01 00:06:00+00:00,0.003254,0.002733,3805
2,1970-01-01 00:07:00+00:00,0.003070,0.002770,4167
3,1970-01-01 00:08:00+00:00,0.001950,0.001823,4338
4,1970-01-01 00:09:00+00:00,0.001689,0.001468,5545


In [15]:
import sys
# Add the current directory to the system path to ensure Colab can find your local .py files
sys.path.append('.')


# Load the Simulator Environment

In [ ]:
from autoscaling_env import AutoScalingEnv
env = AutoScalingEnv(df_usage)


print('Resetting the environment...')
initial_state, initial_info = env.reset()
print(f'Initial State: {initial_state}')
print(f'Initial Info: {initial_info}')

import random
action = random.randint(0, env.action_space.n - 1)
print(f'Taking random action: {action}')

next_state, reward, terminated, truncated, info = env.step(action)

print(f'\nAfter one step:')
print(f'Next State: {next_state}')
print(f'Reward: {reward}')
print(f'Terminated: {terminated}')
print(f'Truncated: {truncated}')
print(f'Info: {info}')

Resetting the environment...
Initial State: [6.6225836e-03 4.9124165e-03 1.0000000e+01]
Initial Info: {'initial_capacity': 10}
Taking random action: 1

After one step:
Next State: [3.2542923e-03 2.7331815e-03 1.0000000e+01]
Reward: -60.80811775319542
Terminated: False
Truncated: False
Info: {'current_capacity': 10, 'utilization': np.float64(6.308010704835946), 'estimated_total_cpu_load': np.float64(63.080107048359466), 'reward_components': {'cost_penalty': -0.02, 'sla_penalty': np.float64(-55.080107048359466), 'util_deviation_penalty': np.float64(-5.708010704835947)}}


#DQN Experiment

In [16]:
from DQN_Utils import train_agent, evaluate_agent, QNetwork, ReplayBuffer
from DQN_Agent import DQNAgent
import numpy as np
import sys
import matplotlib.pyplot as plt # Import missing plt for plotting
import torch # Import torch to define the device

# 1. Initialize the AutoScalingEnv environment for the DQN agent
env_dqn = AutoScalingEnv(df_usage)

# 2. Instantiate the DQNAgent
agent_dqn = DQNAgent(
    observation_space_shape=env_dqn.observation_space.shape,
    action_space_size=env_dqn.action_space.n,
    seed=0
)

# 3. Define the training hyperparameters for the DQN agent
num_episodes_dqn = 500
max_steps_per_episode_dqn = 1000
epsilon_start_dqn = 1.0
epsilon_end_dqn = 0.01
epsilon_decay_dqn = 0.995

# 4. Call the train_agent function
dqn_training_scores = train_agent(
    agent_dqn,
    env_dqn,
    num_episodes_dqn,
    max_steps_per_episode_dqn,
    epsilon_start_dqn,
    epsilon_end_dqn,
    epsilon_decay_dqn
)

print("DQN Training Scores:", dqn_training_scores[:5]) # Display first 5 scores for brevity

# 5. Define the evaluation hyperparameters for the DQN agent
num_evaluation_episodes_dqn = 10
max_steps_per_evaluation_episode_dqn = 1000

# 6. Call the evaluate_agent function
dqn_eval_scores, evaluation_results_df_dqn = evaluate_agent(
    agent_dqn,
    env_dqn,
    num_evaluation_episodes_dqn,
    max_steps_per_evaluation_episode_dqn
)

# 7. Create a figure and a set of subplots
fig, axes = plt.subplots(nrows=3, ncols=1, figsize=(12, 15), sharex=True)

# 8. Plot Current Capacity and Estimated Total CPU Load
axes[0].plot(evaluation_results_df_dqn.index, evaluation_results_df_dqn['current_capacity'], label='Current Capacity')
axes[0].plot(evaluation_results_df_dqn.index, evaluation_results_df_dqn['demand_cpu'], label='CPU Demand')
axes[0].set_ylabel('Capacity / Demand')
axes[0].set_title('DQN Agent Evaluation Results - Capacity and CPU Demand')
axes[0].legend()
axes[0].grid(True)

# 9. Plot Utilization and Target Utilization
axes[1].plot(evaluation_results_df_dqn.index, evaluation_results_df_dqn['utilization'], label='Utilization')
axes[1].axhline(y=env_dqn.target_utilization, color='g', linestyle='--', label='Target Utilization')
axes[1].axhline(y=env_dqn.sla_threshold, color='r', linestyle='--', label='SLA Threshold')
axes[1].set_ylabel('Utilization')
axes[1].legend()
axes[1].grid(True)

# 10. Plot Reward
axes[2].plot(evaluation_results_df_dqn.index, evaluation_results_df_dqn['reward'], label='Reward')
axes[2].set_xlabel('Step')
axes[2].set_ylabel('Reward')
axes[2].legend()
axes[2].grid(True)

# 11. Adjust layout and display plots
plt.tight_layout()
plt.show()

NameError: name 'device' is not defined